# Software Engineering — First Principles Reference

**AIcademy Knowledgebase | Software Engineer Track**

**Sections:**
- [SE-1: Git Workflows](#se-1)
- [SE-2: Writing Clean Code](#se-2)
- [SE-3: Object-Oriented Programming](#se-3)
- [SE-4: Testing with pytest](#se-4)
- [SE-5: REST APIs with FastAPI](#se-5)
- [SE-6: Reading Documentation](#se-6)

---
## SE-1: Git Workflows <a id='se-1'></a>

### Why version control?

Without version control, you save one file and pray. With Git, you have a complete history of every change, who made it, and why. You can always go back. You can work with others without overwriting each other's work.

### The core workflow

```bash
# Check what's changed
git status

# See the actual differences
git diff

# Stage changes (tell git what to include in the next snapshot)
git add filename.py       # specific file
git add .                 # all changed files

# Commit — take the snapshot with a message
git commit -m "add login functionality"

# Upload to GitHub
git push

# Download latest changes from GitHub
git pull
```

### Branches — working without breaking things

A **branch** is a parallel version of your code. You create a branch to work on a new feature, and when it's ready, you merge it back into the main branch.

**Why:** If something breaks on your branch, the main branch is still safe.

```bash
# Create and switch to a new branch
git checkout -b feature/user-authentication

# Work, commit...

# Switch back to main
git checkout main

# Merge your branch into main
git merge feature/user-authentication
```

### Good commit messages
- Bad: `git commit -m "fix"` — tells you nothing
- Bad: `git commit -m "changed stuff"` — vague
- Good: `git commit -m "fix login redirect when session expires"`
- Good: `git commit -m "add pandas groupby exercise to Stage 4"`

---
## SE-2: Writing Clean Code <a id='se-2'></a>

### The principles

**DRY — Don't Repeat Yourself**
If you write the same code twice, extract it into a function. Three times is definitely a function.

In [ ]:
# Not DRY
print("Name:", user1["name"].upper())
print("Name:", user2["name"].upper())
print("Name:", user3["name"].upper())

# DRY
def format_name(user):
    return f"Name: {user['name'].upper()}"

for user in [user1, user2, user3]:
    print(format_name(user))

**Single Responsibility Principle**
Every function should do one thing. If you're describing a function with "and", it does too much.

In [ ]:
# Doing too much — hard to test, hard to reuse
def process_user(user_data):
    validated = validate_user(user_data)         # validation
    hashed = hash_password(validated["password"]) # password handling
    db.save(validated)                            # database
    send_welcome_email(validated["email"])        # email
    return validated

# Better — each function does one thing
def validate_user(user_data): ...
def hash_password(password): ...
def save_user(user): ...
def send_welcome_email(email): ...

def register_user(user_data):
    validated = validate_user(user_data)
    validated["password"] = hash_password(validated["password"])
    save_user(validated)
    send_welcome_email(validated["email"])

**Naming things well**

In [ ]:
# Bad names — requires reading the code to understand
def calc(d, r):
    return d * (1 - r)

x = calc(100, 0.2)

# Good names — self-documenting
def calculate_discounted_price(original_price, discount_rate):
    return original_price * (1 - discount_rate)

final_price = calculate_discounted_price(original_price=100, discount_rate=0.20)

---
## SE-3: Object-Oriented Programming <a id='se-3'></a>

### The mental model

An **object** bundles related **data** (attributes) and **behavior** (methods) together.

**Analogy:** A car has data (color, speed, fuel level) and behavior (accelerate, brake, refuel). Instead of tracking `car_color`, `car_speed`, `car_fuel` as separate variables, you create a `Car` object that holds all of them together.

### Why OOP?
- **Organization** — related things live together
- **Reusability** — create multiple instances from one blueprint
- **Encapsulation** — hide complexity behind a clean interface

In [ ]:
class BankAccount:
    """
    A class is a blueprint. Instances are the actual objects created from it.
    """
    
    def __init__(self, owner, initial_balance=0):
        # __init__ runs when you create a new instance
        # self refers to the specific instance being created
        self.owner = owner
        self.balance = initial_balance
        self.transactions = []
    
    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self.balance += amount
        self.transactions.append(("deposit", amount))
    
    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount
        self.transactions.append(("withdrawal", amount))
    
    def get_balance(self):
        return self.balance
    
    def __str__(self):
        # Special method — controls what print() shows
        return f"BankAccount({self.owner}, balance=${self.balance})"


# Create instances (objects) from the class
alex_account = BankAccount("Alex", initial_balance=500)
jordan_account = BankAccount("Jordan")

alex_account.deposit(200)
alex_account.withdraw(50)
print(alex_account)                  # BankAccount(Alex, balance=$650)
print(alex_account.get_balance())    # 650
print(alex_account.transactions)     # [('deposit', 200), ('withdrawal', 50)]

In [ ]:
# Inheritance — a subclass inherits everything from the parent and can extend it
class SavingsAccount(BankAccount):
    def __init__(self, owner, initial_balance=0, interest_rate=0.02):
        super().__init__(owner, initial_balance)  # call parent's __init__
        self.interest_rate = interest_rate
    
    def apply_interest(self):
        interest = self.balance * self.interest_rate
        self.deposit(interest)
        return interest

savings = SavingsAccount("Alex", 1000, interest_rate=0.05)
earned = savings.apply_interest()
print(f"Earned ${earned:.2f} in interest")
print(savings)  # SavingsAccount inherits BankAccount's __str__

---
## SE-4: Testing with pytest <a id='se-4'></a>

### Why tests?

Tests are code that checks your code. Once written, you can run them in seconds any time you make changes — giving you confidence you didn't break anything.

**Without tests:** "I think this works..." → change something → "I hope I didn't break anything"
**With tests:** Change something → run tests → instantly know if anything broke

In [ ]:
# The function you want to test
def calculate_discounted_price(original_price, discount_rate):
    if discount_rate < 0 or discount_rate > 1:
        raise ValueError("Discount rate must be between 0 and 1")
    return round(original_price * (1 - discount_rate), 2)


# Test functions must start with test_
def test_basic_discount():
    assert calculate_discounted_price(100, 0.20) == 80.0

def test_no_discount():
    assert calculate_discounted_price(100, 0) == 100.0

def test_full_discount():
    assert calculate_discounted_price(100, 1.0) == 0.0

def test_invalid_discount_raises_error():
    import pytest
    with pytest.raises(ValueError):
        calculate_discounted_price(100, 1.5)  # > 1 is invalid

# Run from terminal: pytest test_file.py -v

### What to test
- **Happy path** — normal input, expected output
- **Edge cases** — empty input, zero, negative numbers, None
- **Error cases** — invalid input should raise specific exceptions

---
## SE-5: REST APIs with FastAPI <a id='se-5'></a>

### What is a REST API?

A **REST API** is a way for programs to communicate over the internet using HTTP. Your Python code can be a server that other programs (web apps, mobile apps, other scripts) talk to.

**HTTP methods:**
| Method | Purpose | Analogy |
|---|---|---|
| GET | Read data | Looking something up |
| POST | Create new data | Filling out a form |
| PUT | Replace existing data | Updating your entire profile |
| PATCH | Partially update data | Changing just your email |
| DELETE | Remove data | Deleting an account |

In [ ]:
# pip install fastapi uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()

# In-memory database (replace with a real DB in production)
users = {
    1: {"id": 1, "name": "Alex", "email": "alex@example.com"},
    2: {"id": 2, "name": "Jordan", "email": "jordan@example.com"}
}

# GET — read all users
@app.get("/users")
def get_all_users():
    return list(users.values())

# GET — read one user by ID
@app.get("/users/{user_id}")
def get_user(user_id: int):
    if user_id not in users:
        raise HTTPException(status_code=404, detail="User not found")
    return users[user_id]

# POST — create a new user
class UserCreate(BaseModel):
    name: str
    email: str

@app.post("/users")
def create_user(user: UserCreate):
    new_id = max(users.keys()) + 1
    new_user = {"id": new_id, "name": user.name, "email": user.email}
    users[new_id] = new_user
    return new_user

# Run with: uvicorn filename:app --reload
# API docs auto-generated at: http://localhost:8000/docs

---
## SE-6: Reading Documentation <a id='se-6'></a>

### The skill no one teaches

Documentation is written by the people who built the tool. It is the source of truth. Being able to read docs efficiently separates engineers who are always asking others from engineers who can figure things out independently.

### How to navigate docs quickly

1. **Start with the quickstart** — not the full reference. Get something working first.
2. **Look for the API reference** — every function, parameter, and return value listed. Use `Ctrl+F`.
3. **Read the examples** — copy one, run it, then modify it.
4. **Check the changelog** — if something broke, the version might have changed behavior.

### How to read a function signature

In [ ]:
# How to interpret a Python function signature:
# pd.read_csv(filepath_or_buffer, sep=',', header='infer', names=None, ...)
#
# - filepath_or_buffer: required (no default)
# - sep=',': optional — defaults to comma
# - header='infer': optional — auto-detects header row
# - names=None: optional — column names, defaults to first row

import pandas as pd

# You can always check what a function accepts using help()
help(pd.read_csv)

# Or in Jupyter, use ? for a quick reference
# pd.read_csv?